# Quick Check Slice



## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path

# Add parent directory to Python path to import modules
repo_root = Path.cwd().parent
# No longer needed - using installed package

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

### Install fabric-generic-cluster

Install from pypi (https://pypi.org/project/fabric-generic-cluster) or git repo (https://github.com/mcevik0/fabric-generic-cluster)

In [ ]:
def install_from_pypi():
    print("Installing from PyPI...")
    !pip install --upgrade fabric-generic-cluster --quiet

In [ ]:
def install_from_local():
    print("Installing from local git clone...")
    !rm -rf fabric-generic-cluster
    !git clone https://github.com/mcevik0/fabric-generic-cluster.git
    %cd fabric-generic-cluster
    !pip install -e . --quiet
    %cd ..

#### Select method and install

Install from pypi - recommended.

In [ ]:
install_method = "pypi"   # or "local"

if install_method == "pypi":
    install_from_pypi()
elif install_method == "local":
    install_from_local()
else:
    print("Unknown installation method:", install_method)

### Configure

In [ ]:
# Import modules

from fabric_generic_cluster import load_topology_from_yaml_file, SiteTopology
from fabric_generic_cluster import deployment as sd
from fabric_generic_cluster import network_config as snc
from fabric_generic_cluster import ssh_setup as ssh
from fabric_generic_cluster import ansible_setup as ansible
from fabric_generic_cluster import selinux_management as selinux


print("✅ Modules imported successfully")

## 2. Check Existing Slices

In [ ]:
sd.check_slices()

## 3. Set Slice Deployment Attributes

In [ ]:
# Define YAML directory

YAML_DIR = repo_root / "model"
print(f"✅ YAML directory: {YAML_DIR}")

In [ ]:
# Define your base slice name

slice_name = "slice"

# Site topology YAML file

site_topology_yaml = "../model/m7.yml"

## 4. Load and Validate Topology


In [ ]:
# Load and validate topology (raises ValidationError if invalid)

try:
    topology = load_topology_from_yaml_file(site_topology_yaml)
    print("✅ Topology loaded and validated successfully!")
    print(f"   Nodes: {len(topology.site_topology_nodes.nodes)}")
    print(f"   Networks: {len(topology.site_topology_networks.networks)}")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

## 5. Explore Topology


In [ ]:
# Print topology summary using type-safe access

print("\n🔷 Topology Summary:\n")

for node in topology.site_topology_nodes.iter_nodes():
    print(f"Node: {node.hostname}")
    print(f"  Site: {node.site}")
    print(f"  Resources: {node.capacity.cpu} cores, {node.capacity.ram}GB RAM")
    print(f"  OS: {node.capacity.os}")
    
    # Check OpenStack roles
    roles = node.specific.openstack
    active_roles = []
    if roles.is_control(): active_roles.append("control")
    if roles.is_network(): active_roles.append("network")
    if roles.is_compute(): active_roles.append("compute")
    if roles.is_storage(): active_roles.append("storage")
    
    if active_roles:
        print(f"  OpenStack roles: {', '.join(active_roles)}")
    
    # Print network interfaces
    for nic_name, iface_name, iface in node.get_all_interfaces():
        ipv4 = iface.get_ipv4_address() or "No IPv4"
        print(f"  └─ {nic_name}.{iface_name}: {iface.binding} ({ipv4})")
    
    print()

## 8. Observe Slice Attributes

In [ ]:
try:
    slice = sd.get_slice(slice_name)
    slice.show()
    slice.list_nodes()
    slice.list_networks()
    slice.list_interfaces()
except Exception as e:
    print(f"Exception: {e}")

## 9. Set Source Node and Network for Verification



In [ ]:
# Get all nodes as a list, then take first
all_nodes = list(topology.site_topology_nodes.iter_nodes())
first_node = all_nodes[0]
iface1 = first_node.pci.network['nic1'].interfaces['iface1']
binding = iface1.binding

# Alternative: Access by dictionary key
#first_node = topology.site_topology_nodes.nodes['node1']
#print(first_node.name)

source_hostname = first_node.name
network_name = binding

use_ipv6 = False

## 10. Test Network Connectivity

In [ ]:
# <FIXME> Error handling

# Test connectivity 
try: 
    results = snc.ping_network_from_node(
        slice, 
        topology, 
        source_hostname=source_hostname, 
        network_name=network_name,
        use_ipv6=use_ipv6
    )
    if all(results.values()):
        print("\n✅ All ping tests passed!\n")
    else:
        print("\n⚠️  Some ping tests failed\n")
except Exception as e:
    print(f"\n❌ Test network connectivity failed: {e}\n")


## 10. Verify SSH Access

In [ ]:
# Test SSH connectivity

ssh_results = ssh.verify_ssh_access(
    slice,
    topology,
    source_hostname=source_hostname,
    network_name=network_name,
    use_ipv6=use_ipv6
)

if all(ssh_results.values()):
    print("\n✅ All SSH connections successful!\n")
else:
    print("\n⚠️  Some SSH connections failed\n")